In [ ]:
'''
Candidate generation (RA/Dec boundary stage)

This runs before the model. Keep it outside PyTorch.
'''
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

def get_gaia_candidates(
    xmm_ra, xmm_dec, xmm_poserr_arcsec,
    gaia_ra, gaia_dec,
    k_sigma=3.0
):
    """
    Returns indices of Gaia sources within k-sigma XMM error radius + a dedicated NULL index.
    """
    xmm_coord = SkyCoord(ra=xmm_ra*u.deg, dec=xmm_dec*u.deg)
    gaia_coord = SkyCoord(ra=gaia_ra*u.deg, dec=gaia_dec*u.deg)

    sep = xmm_coord.separation(gaia_coord).arcsec
    radius = k_sigma * xmm_poserr_arcsec
    sep = np.atleast_1d(sep)

    candidate_indices = np.where(sep <= radius)[0]
    candidate_seps = sep[candidate_indices]

    final_indices = np.append(candidate_indices, -1) 
    final_seps = np.append(candidate_seps, radius)

    return final_indices, final_seps

'''
This function defines the valid observation space.
Everything downstream assumes this has already been applied.
'''

Matching Heads

In [ ]:
# # Distance based similarity (no parameters)
# import torch
# import torch.nn.functional as F

# def distance_scores(z_xmm, z_gaia):
#     """
#     z_xmm:  (D,)
#     z_gaia: (K, D)
#     returns: (K,)
#     """
#     return -torch.sum((z_gaia - z_xmm.unsqueeze(0))**2, dim=1)


In [ ]:
#Bilinearity search (Default)

import torch
import torch.nn as nn
import torch.nn.functional as F

class BilinearMatcher(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.W = nn.Parameter(torch.randn(dim, dim) * 0.01)

    def forward(self, z_xmm, z_gaia):
        """
        z_xmm:  (D,)
        z_gaia: (K, D)
        returns: (K,)
        """
        zx = z_xmm @ self.W          # (D,)
        return torch.sum(z_gaia * zx.unsqueeze(0), dim=1)


In [ ]:
# Optional: add relative separation safely, If you want angular separation at this stage. It's a good idea

def add_positional_bias(scores, sep_arcsec, sigma_arcsec, alpha=1.0):
    """
    scores:      (K,)
    sep_arcsec:  numpy or torch (K,)
    sigma_arcsec: scalar
    """
    d2 = (sep_arcsec / sigma_arcsec)**2
    d2 = torch.as_tensor(d2, device=scores.device, dtype=scores.dtype)
    return scores - alpha * d2


In [ ]:
def listwise_loss(scores, true_index):
    """
    scores: (K,)
    true_index: int
    """
    log_probs = F.log_softmax(scores, dim=0)
    return -log_probs[true_index]


In [ ]:
def train_matcher(
    xmm_encoder,
    gaia_encoder,
    matcher,
    dataloader,
    optimizer,
    device
):
    xmm_encoder.eval()
    gaia_encoder.eval()
    matcher.train()

    total_loss = 0.0

    for batch in dataloader:
        """
        batch must contain:
        - xmm_features
        - gaia_features_list (list of tensors, variable K)
        - sep_arcsec_list
        - true_index
        """
        xmm_feat = batch["xmm_features"].to(device)
        gaia_feat = batch["gaia_features"].to(device)
        true_idx = batch["true_index"]

        with torch.no_grad():
            z_xmm = xmm_encoder(xmm_feat)           # (D,)
            z_gaia = gaia_encoder(gaia_feat)        # (K, D)

        scores = matcher(z_xmm, z_gaia)

        if "sep_arcsec" in batch:
            scores = add_positional_bias(
                scores,
                batch["sep_arcsec"],
                batch["xmm_poserr_arcsec"]
            )

        loss = listwise_loss(scores, true_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)


In [ ]:
# Evaluation metrics
def top1_accuracy(scores, true_index):
    return int(torch.argmax(scores).item() == true_index)

def reciprocal_rank(scores, true_index):
    ranking = torch.argsort(scores, descending=True)
    rank = (ranking == true_index).nonzero(as_tuple=True)[0].item() + 1
    return 1.0 / rank


#----------------------------------------------------------------------------------------------------------------------------------------#

In [ ]:
'''
Candidate generation (RA/Dec boundary stage)

This runs before the model. Keep it outside PyTorch.
'''
import numpy as np
from astropy.coordinates import SkyCoord
import astropy.units as u

def get_gaia_candidates(
    xmm_ra, xmm_dec, xmm_poserr_arcsec,
    gaia_ra, gaia_dec,
    k_sigma=3.0
):
    """
    Returns indices of Gaia sources within k-sigma XMM error radius + a dedicated NULL index.
    """
    xmm_coord = SkyCoord(ra=xmm_ra*u.deg, dec=xmm_dec*u.deg)
    gaia_coord = SkyCoord(ra=gaia_ra*u.deg, dec=gaia_dec*u.deg)

    sep = xmm_coord.separation(gaia_coord).arcsec
    radius = k_sigma * xmm_poserr_arcsec
    sep = np.atleast_1d(sep)

    candidate_indices = np.where(sep <= radius)[0]
    candidate_seps = sep[candidate_indices]

    final_indices = np.append(candidate_indices, -1) 
    final_seps = np.append(candidate_seps, radius)

    return final_indices, final_seps

'''
This function defines the valid observation space.
Everything downstream assumes this has already been applied.
'''

In [ ]:
class XMMGaiaMatcher(nn.Module):
    """
    Conditional listwise matcher with optional NULL candidate.
    """

    def __init__(
        self,
        embed_dim=64,
        mode="bilinear",     # "distance" or "bilinear"
        use_null=True
    ):
        super().__init__()

        assert mode in ("distance", "bilinear")
        self.mode = mode
        self.use_null = use_null

        if mode == "bilinear":
            self.W = nn.Parameter(torch.randn(embed_dim, embed_dim) * 0.01)

        if use_null:
            # Learnable bias for "no counterpart"
            self.null_bias = nn.Parameter(torch.tensor(0.0))

    def score_pairs(self, z_xmm, z_gaia):
        """
        z_xmm:  (D,)
        z_gaia: (K, D)
        returns: (K,)
        """
        if self.mode == "distance":
            return -torch.sum((z_gaia - z_xmm.unsqueeze(0))**2, dim=1)

        # bilinear
        zx = z_xmm @ self.W                  # (D,)
        return torch.sum(z_gaia * zx.unsqueeze(0), dim=1)

    def forward(
        self,
        z_xmm,
        z_gaia,
        sep_arcsec=None,
        xmm_poserr_arcsec=None,
        alpha=1.0
    ):
        """
        Returns:
            scores: (K [+1 if NULL])
        """
        scores = self.score_pairs(z_xmm, z_gaia)

        # Optional positional penalty
        if sep_arcsec is not None:
            d2 = (sep_arcsec / xmm_poserr_arcsec)**2
            d2 = torch.as_tensor(d2, device=scores.device, dtype=scores.dtype)
            scores = scores - alpha * d2

        # Append NULL candidate
        if self.use_null:
            scores = torch.cat(
                [scores, self.null_bias.view(1)],
                dim=0
            )

        return scores


In [6]:
from src.inference.encoder import xmm_encoder, gaia_encoder
import torch

base_dir = r"C:/Raj_Stuff/Coding/VScode/Workspaces/Astro-ML"
xmm_encoder_path = f"{base_dir}/models/encoders/1e-3/simclr_xmm_3_frozen.pth"
gaia_encoder_path = f"{base_dir}/models/encoders/1e-3/simclr_gaia_3_frozen.pth"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

xmm_encoder = xmm_encoder(xmm_encoder_path)
gaia_encoder = gaia_encoder(gaia_encoder_path)
xmm_encoder.to(device)
gaia_encoder.to(device)

for p in xmm_encoder.parameters():
    p.requires_grad = False

for p in gaia_encoder.parameters():
    p.requires_grad = False

xmm_encoder.eval()
gaia_encoder.eval()

with torch.no_grad():
    z_xmm = xmm_encoder(torch.randn(1, 6).to(device))
    z_gaia = gaia_encoder(torch.randn(5, 8).to(device))

print(z_xmm.shape)   # (1, 64)
print(z_gaia.shape)  # (5, 64)


torch.Size([1, 64])
torch.Size([5, 64])


In [ ]:
def match_probabilities(scores):
    """
    scores: (K [+1])
    returns: probabilities with same shape
    """
    return F.softmax(scores, dim=0)

def listwise_loss(scores, true_index):
    """
    scores: (K [+1])
    true_index:
        - int in [0, K-1] for real Gaia match
        - int == K for NULL (no counterpart)
    """
    log_probs = F.log_softmax(scores, dim=0)
    return -log_probs[true_index]


In [ ]:
def train_epoch(
    xmm_encoder,
    gaia_encoder,
    matcher,
    dataloader,
    optimizer,
    device = "cuda" if torch.cuda.is_available() else "cpu"
):
    xmm_encoder.eval()
    gaia_encoder.eval()
    matcher.train()

    total_loss = 0.0

    for batch in dataloader:
        """
        batch fields:
        - xmm_features           (Dx,)
        - gaia_features          (K, Dg)
        - sep_arcsec             (K,)
        - xmm_poserr_arcsec      scalar
        - true_index             int (K if NULL)
        """
        xmm_feat = batch["xmm_features"].to(device)
        gaia_feat = batch["gaia_features"].to(device)
        true_idx = batch["true_index"]

        with torch.no_grad():
            z_xmm = xmm_encoder(xmm_feat)        # (D,)
            z_gaia = gaia_encoder(gaia_feat)     # (K, D)

        scores = matcher(
            z_xmm,
            z_gaia,
            sep_arcsec=batch.get("sep_arcsec"),
            xmm_poserr_arcsec=batch.get("xmm_poserr_arcsec")
        )

        loss = listwise_loss(scores, true_idx)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    return total_loss / len(dataloader)

In [ ]:
probs = match_probabilities(scores)

best = torch.argmax(probs).item()

if matcher.use_null and best == len(probs) - 1:
    decision = "NO_COUNTERPART"
else:
    decision = f"GAIA_{best}"